# 정치 테마주 분석 시스템 — 첫 번째 테스트 실행 결과

**실행일**: 2026-03-27  
**선거까지**: D-68 (제9회 전국동시지방선거 2026.06.03)  
**현재 단계**: 선거 3~1개월 전 (공천 확정 → 최고 과열 구간)  
**시그널**: 고점 주의, 분할 매도 고려

---

## 테스트 체크리스트

| 항목 | 결과 |
|---|---|
| pykrx 설치 및 KRX 데이터 수집 | ✅ 정상 |
| election_calendar.yaml 로드 | ✅ 정상 |
| politician_stock_map.yaml 로드 | ✅ 정상 (28개 종목) |
| D-day 자동 계산 | ✅ D-68 |
| 선거 단계 자동 판별 | ✅ 3~1개월 전 단계 |
| 전체 종목 스크리닝 | ✅ 28개 종목 완료 |
| JSON 결과 아카이빙 | ✅ data/processed/test_run_2026-03-27.json |

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import json

from collectors.stock_collector import StockCollector
from analyzers.theme_mapper import ThemeMapper
from collectors.poll_collector import PollCollector

sc = StockCollector()
tm = ThemeMapper()
pc = PollCollector()

## 1. 선거 현황

In [ ]:
phase = pc.get_election_phase()
print(pc.summarize_election_status())
print()
print('[추적 후보 목록]')
candidates = pc.get_tracking_candidates()
df_cands = pd.DataFrame(candidates)
df_cands

## 2. 오늘의 정치 테마주 스크리닝 (2026-03-27)

In [ ]:
tickers = tm.get_all_tickers()
results = sc.screen_theme_stocks(tickers, surge_ratio=1.5)

# 데이터프레임 변환
df = pd.DataFrame(results)
df['surge'] = df['surge'].astype(bool)
df = df.sort_values('change_pct', ascending=False)

print(f'스크리닝 종목 수: {len(df)}개')
print(f'상승 종목: {(df.change_pct > 0).sum()}개')
print(f'하락 종목: {(df.change_pct < 0).sum()}개')
print(f'거래량 급등 종목: {df.surge.sum()}개')

df[['name','ticker','close','change_pct','ratio','surge']]

## 3. 등락률 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

df_plot = df[df['close'] > 0].sort_values('change_pct')
colors = ['#e74c3c' if x > 0 else '#3498db' for x in df_plot['change_pct']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(df_plot['name'], df_plot['change_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('등락률 (%)')
ax.set_title(f'정치 테마주 등락률 (2026-03-27) | D-68 지방선거')
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, df_plot['change_pct']):
    ax.text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/chart_2026-03-27.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: data/processed/chart_2026-03-27.png')

## 4. 주요 발견사항 (2026-03-27)

### 시장 전반
- 오늘 정치 테마주 28개 중 **25개 하락** (시장 전반 조정 국면)
- 거래량은 전반적으로 평균 대비 낮음 (0.1~0.9배)
- 지방선거 D-68일, 현재 **공천 확정 단계** → 이론적으로는 2차 급등 구간

### 주목 종목
| 종목 | 등락 | 특이사항 |
|---|---|---|
| 대우건설 | +5.53% | 유일한 큰 폭 상승, SOC 테마 |
| 효성중공업 | -8.67% | 원전 테마 조정 |
| 삼성전기 | -7.66% | 반도체 테마 조정 |

### 다음 모니터링 포인트
- 박찬대 인천시장 단수 공천 확정 (2026.03.04) → 인천 관련주 반응 추적 필요
- 서울시장 민주당 경선 결과 → 정원오 관련주 (에스제이그룹 등) 시그널
- 경기지사 경선 → 5인 중 승자 확정 시 급등 예상

In [ ]:
# 테스트 결과 최종 저장
import json, datetime
from pathlib import Path

class SafeEncoder(json.JSONEncoder):
    def default(self, o):
        if hasattr(o, 'item'): return o.item()
        return str(o)

output = {
    'test_date': '2026-03-27',
    'election_phase': phase,
    'total_tracked': len(tickers),
    'screening_results': [
        {**r, 'surge': bool(r.get('surge', False))} for r in results
    ]
}

Path('../data/processed').mkdir(parents=True, exist_ok=True)
with open('../data/processed/test_run_2026-03-27.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, cls=SafeEncoder)

print('아카이빙 완료: data/processed/test_run_2026-03-27.json')